# k-NN — Nearest Neighbor Pattern Classification (1967)

_Papers · Classical ML_

**Classify a point by copying the label of its single closest training example — and that simple rule is provably never worse than twice the best possible error.**

---

This notebook is a practice scaffold for the **k-NN — Nearest Neighbor Pattern Classification (1967)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
torch.manual_seed(0)

# --- 0. Sanity-check the lesson's worked example by hand. ---
Xtr = torch.tensor([[0.0, 0.0],
                    [3.0, 4.0],
                    [1.0, 0.0]])
ytr = torch.tensor([0, 1, 0])
q   = torch.tensor([[2.0, 0.0]])               # one query point

d2_hand = ((q - Xtr) ** 2).sum(dim=1)          # squared dists: (2-0)^2.. etc
print("worked d^2:", d2_hand.tolist())         # -> [4.0, 17.0, 1.0]
nn_idx = torch.argmin(d2_hand)
print("nearest idx:", nn_idx.item(), " predicted label:", ytr[nn_idx].item())
# -> nearest idx: 2  predicted label: 0


# --- 1. The 1-NN rule, built from scratch (NO torch.cdist). ---
def knn_predict_scratch(Xtr, ytr, Xte):
    # ||a-b||^2 = ||a||^2 - 2 a.b + ||b||^2 , broadcast over all (test, train) pairs
    te2   = (Xte ** 2).sum(dim=1, keepdim=True)      # (T,1)  query squared norms
    tr2   = (Xtr ** 2).sum(dim=1, keepdim=True).T    # (1,N)  train squared norms
    cross = Xte @ Xtr.T                              # (T,N)  inner products
    d2    = te2 - 2 * cross + tr2                     # (T,N)  squared distances
    nn    = torch.argmin(d2, dim=1)                   # nearest train index per query
    return ytr[nn], d2                                # copy the neighbor's label


# --- 2. Reference oracle: torch.cdist + argmin (only used to check ourselves). ---
def knn_predict_oracle(Xtr, ytr, Xte):
    d  = torch.cdist(Xte, Xtr)                        # (T,N) Euclidean distances
    nn = torch.argmin(d, dim=1)
    return ytr[nn]


# --- 3. Toy data, then VERIFY mine == oracle. ---
N, T, D, K = 200, 50, 5, 3
Xtr_r = torch.randn(N, D)
ytr_r = torch.randint(0, K, (N,))
Xte_r = torch.randn(T, D)

pred_mine, d2 = knn_predict_scratch(Xtr_r, ytr_r, Xte_r)
pred_ref      = knn_predict_oracle(Xtr_r, ytr_r, Xte_r)

d_ref2 = torch.cdist(Xte_r, Xtr_r) ** 2              # oracle squared distances
assert torch.allclose(d2, d_ref2, atol=1e-4), "distance matrix mismatch!"
assert bool((pred_mine == pred_ref).all()),  "label mismatch!"
print("allclose(d2, cdist^2):", torch.allclose(d2, d_ref2, atol=1e-4))   # True
print("exact label match    :", bool((pred_mine == pred_ref).all()))     # True
print("first 8 preds mine  :", pred_mine[:8].tolist())
print("first 8 preds oracle:", pred_ref[:8].tolist())
# allclose(d2, cdist^2): True
# exact label match    : True
# first 8 preds mine  : [2, 0, 1, 1, 1, 0, 0, 0]
# first 8 preds oracle: [2, 0, 1, 1, 1, 0, 0, 0]
# My hand-built 1-NN rule IS torch's cdist+argmin -- that's the Track A payoff.

## Visualize the data & results

_Does the 1-NN error rate really sit between the Bayes error R* and 2R*, as Cover & Hart prove?_

In [ ]:
import torch
torch.manual_seed(0)

# Two 1-D Gaussian classes, equal priors. R* known in closed form;
# 1-NN error measured by simulation. Show R* <= R_NN <= 2R* (Cover-Hart).
def normal_pdf(x, mu, sd):
    return torch.exp(-0.5*((x-mu)/sd)**2) / (sd*(2*torch.pi)**0.5)

def bayes_error(mu0, mu1, sd, grid):
    p0 = 0.5*normal_pdf(grid, mu0, sd)            # joint density, class 0 (prior 1/2)
    p1 = 0.5*normal_pdf(grid, mu1, sd)            # joint density, class 1
    px = p0 + p1
    post_min = torch.minimum(p0, p1) / px         # min posterior at each x
    dx = grid[1] - grid[0]
    return float((post_min * px * dx).sum())      # R* = E[min posterior]

def nn_error(mu0, mu1, sd, n=4000, ntest=4000, seed=0):
    g = torch.Generator().manual_seed(seed)
    def sample(m):
        y = torch.randint(0, 2, (m,), generator=g)
        x = torch.where(y == 0, torch.normal(mu0, sd, (m,), generator=g),
                                torch.normal(mu1, sd, (m,), generator=g))
        return x.unsqueeze(1), y
    Xtr, ytr = sample(n); Xte, yte = sample(ntest)
    nn = torch.argmin(torch.cdist(Xte, Xtr), dim=1)   # 1-NN
    return float((ytr[nn] != yte).float().mean())

grid = torch.linspace(-12, 12, 20000)
for sep in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]:
    mu0, mu1 = -sep/2, sep/2
    Rstar = bayes_error(mu0, mu1, 1.0, grid)
    Rnn   = nn_error(mu0, mu1, 1.0)
    print(f"R*={Rstar:.4f}  R_NN={Rnn:.4f}  2R*={2*Rstar:.4f}  in-band={Rstar<=Rnn<=2*Rstar+0.01}")
# R*=0.4012  R_NN=0.4740  2R*=0.8024  in-band=True
# R*=0.3085  R_NN=0.4040  2R*=0.6169  in-band=True
# R*=0.2266  R_NN=0.3158  2R*=0.4531  in-band=True
# R*=0.1586  R_NN=0.2305  2R*=0.3172  in-band=True
# R*=0.1056  R_NN=0.1540  2R*=0.2112  in-band=True
# R*=0.0668  R_NN=0.1010  2R*=0.1336  in-band=True
# R*=0.0227  R_NN=0.0345  2R*=0.0455  in-band=True
# R*=0.0062  R_NN=0.0093  2R*=0.0124  in-band=True

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Apply the bound. On a two-class problem the optimal (Bayes) classifier errs $5\%$ of the
            time, so $R^* = 0.05$. With a very large training set, what is the tightest upper bound the
            Cover-Hart theorem gives for the nearest-neighbor error rate $R$, and what is the loose "twice"
            bound?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Plug $R^* = 0.05$ into the tight two-class bound $2R^*(1-R^*)$. — _Eqn. 13's upper bound is $2R^*(1-R^*)$, the tightest the theorem provides for $M=2$._
- Compute: $2 \times 0.05 \times (1 - 0.05) = 2 \times 0.05 \times 0.95 = 0.095$. — _That is the guaranteed ceiling on the asymptotic NN error._
- Compare with the loose bound $2R^* = 2 \times 0.05 = 0.10$. — _The "$\le 2R^*$" headline drops the $(1-R^*)$ factor, giving a slightly looser $10\%$._

**Answer:** The tight bound is $2R^*(1-R^*) = 2(0.05)(0.95) = \mathbf{0.095}$ (i.e. $9.5\%$); the loose
                 "twice" bound is $2R^* = \mathbf{0.10}$ ($10\%$). So a memorize-the-nearest-point rule, given
                 enough data, is guaranteed to err at most about $9.5\%$ of the time when the best possible
                 rule errs $5\%$.

</details>

**Problem 2.** The ablation: NN vs Bayes as the classes separate. Take two 1-D Gaussian classes with equal
            priors. As you push the class means apart, the Bayes error $R^*$ shrinks toward $0$. What happens
            to the gap between the NN error $R$ and $R^*$, and does $R$ ever break the $2R^*$ ceiling?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Reason from $R = 2R^*(1-R^*) - 2\,\mathrm{Var}\,r^*$: as separation grows, $R^* \to 0$, so the upper expression $2R^*(1-R^*) \to 2R^*$ and both collapse toward $0$. — _Small $R^*$ forces small $R$; the multiplicative bound shrinks the absolute gap._
- Predict $R$ stays inside $[R^*,\,2R^*]$ at every separation, and that $R/R^*$ approaches a constant near $1$-to-$2$ as $R^* \to 0$. — _The theorem guarantees the band; the ratio cannot exceed $2$ asymptotically._
- Run the CODEVIZ simulation and read off the (R*, R_NN) pairs to confirm each sits below the $2R^*$ line. — _An empirical check that the proven sandwich holds on real samples._

**Answer:** The absolute gap $R - R^*$ shrinks to $0$ as the classes separate (both errors vanish), and
                 $R$ stays strictly between $R^*$ and $2R^*$ throughout &mdash; never breaking the ceiling. In
                 our run, at well-separated means $R^* \approx 0.006$ gave $R_{NN} \approx 0.009$ (ratio
                 $\approx 1.5$, comfortably under $2$); at barely-separated means $R^* \approx 0.40$ gave
                 $R_{NN} \approx 0.47$, still below $2R^* = 0.80$. The CODEVIZ panel plots exactly this band.

</details>

**Problem 3.** Your worked example had query $x=(2,0)$ with nearest neighbor $x_3=(1,0)$, label $0$. Suppose you
            move the query to $x=(2.6,\,0)$. Recompute the squared distances and the predicted label. Does the
            answer change, and what does the boundary between predictions look like?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Squared distance to $x_1=(0,0)$: $(2.6)^2 = 6.76$. To $x_3=(1,0)$: $(1.6)^2 = 2.56$. To $x_2=(3,4)$: $(0.4)^2 + 16 = 16.16$. — _Same expansion as before; $x_2$ is still far because of its large second coordinate._
- Take the argmin of $[6.76,\,16.16,\,2.56]$: still $x_3$ (distance $2.56$). — _$x_3$ at $(1,0)$ remains the closest along the $x$-axis until the query passes the midpoint toward another point._
- Copy $x_3$'s label: predict class $0$ again. — _The NN rule outputs the nearest point's label; the nearest point is unchanged._

**Answer:** Distances are $[6.76,\,16.16,\,2.56]$, nearest is $x_3$, so the prediction is still
                 class 0. The prediction only flips where the query crosses the perpendicular bisector
                 between two training points of different classes &mdash; that is the NN decision
                 boundary, a stitching-together of such bisectors (a Voronoi diagram).

</details>